# 11 GHSL Reusable Urban Growth Model

## Phase 1: Six-city inventory and modeling preflight

This first notebook phase is deliberately read-only. It verifies the exact HDFS city slugs, all 60 enriched epoch partitions, 4,900 rows per partition, unique grid keys, and schema compatibility before any longitudinal dataset is written.

**Training cities:** Bengaluru, Hyderabad, Pune Metropolitan Region, Chennai Metropolitan Region, Mumbai Metropolitan Region, and Delhi NCR.  
**Held-out city:** Jaipur. Jaipur is not used in this notebook.

The notebook uses `/content/drive/MyDrive/remote_exec.sh` as the single source of truth for the EC2 host, user, SSH key, and SSH options. Google Drive is mounted because it also stores the persistent inventory and modeling artifacts.

Run one cell at a time. Do not skip a failed gate.

In [ ]:
# ============================================================
# CELL 1: MOUNT GOOGLE DRIVE AND LOCAL CONFIGURATION
# ============================================================

from google.colab import drive
from pathlib import Path
import base64
import json
import os
import re
import subprocess
import time

# Google Drive is still required because remote_exec.sh is the
# single source of truth for the current EC2 connection details.
drive.mount(
    "/content/drive",
    force_remount=False,
)

REMOTE_EXEC = Path(
    "/content/drive/MyDrive/remote_exec.sh"
)

LOCAL_ARTIFACT_ROOT = Path(
    "/content/drive/MyDrive/GHSL_Urban_Growth_Model"
)

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(
        "remote_exec.sh was not found at: "
        + str(REMOTE_EXEC)
    )

REMOTE_EXEC.chmod(0o700)

LOCAL_ARTIFACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_EPOCHS = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]

EXPECTED_ROWS_PER_PARTITION = 4900
EXPECTED_CITY_COUNT = 6
EXPECTED_PARTITION_COUNT = 60
EXPECTED_TOTAL_ROWS = 294000

TRAINING_CITY_HINTS = {
    "Bengaluru": [
        "bengaluru",
        "bangalore",
    ],
    "Hyderabad": [
        "hyderabad",
    ],
    "Pune Metropolitan Region": [
        "pune",
    ],
    "Chennai Metropolitan Region": [
        "chennai",
    ],
    "Mumbai Metropolitan Region": [
        "mumbai",
    ],
    "Delhi NCR": [
        "delhi",
        "ncr",
    ],
}

print("Google Drive mounted: PASS")
print("Remote executor:", REMOTE_EXEC)
print("Persistent artifact root:", LOCAL_ARTIFACT_ROOT)
print("Expected cities: 6")
print("Expected enriched partitions: 60")
print("Expected grid-year rows: 294000")
print("Jaipur included in training: NO")
print("HDFS mutation enabled: NO")
print("CELL 1: SUCCESS")

In [ ]:
# ============================================================
# CELL 2: REMOTE_EXEC.SH VALIDATION AND REMOTE HELPER
#
# The EC2 host, user, SSH key and SSH options remain inside
# remote_exec.sh. Nothing is duplicated or hardcoded here.
# ============================================================

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(
        "remote_exec.sh is unavailable: "
        + str(REMOTE_EXEC)
    )

REMOTE_EXEC.chmod(0o700)


def run_remote(
    command,
    timeout=300,
    show_output=True,
):
    result = subprocess.run(
        [
            str(REMOTE_EXEC),
            command,
        ],
        text=True,
        capture_output=True,
        timeout=timeout,
    )

    if show_output and result.stdout:
        print(result.stdout)

    if show_output and result.stderr:
        print("REMOTE STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "Remote command failed with return code "
            + str(result.returncode)
            + "."
        )

    return result


connection_test = run_remote(
    r'''
set -e

echo "REMOTE_EXEC_CONNECTION_SUCCESS"
echo "Remote user: $(whoami)"
echo "Remote host: $(hostname)"
''',
    timeout=90,
    show_output=True,
)

if (
    "REMOTE_EXEC_CONNECTION_SUCCESS"
    not in connection_test.stdout
):
    raise RuntimeError(
        "remote_exec.sh returned successfully but the "
        "connection marker was not found."
    )

print("remote_exec.sh connection: PASS")
print("Hardcoded EC2 host in notebook: NO")
print("Hardcoded SSH key in notebook: NO")
print("CELL 2: SUCCESS")

In [ ]:
# ============================================================
# CELL 3: READ-ONLY EC2/HDFS/SPARK/SEDONA PREFLIGHT
# ============================================================

cmd=r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
echo "============================================================"
echo "SIX-CITY MODELING PREFLIGHT"
echo "============================================================"
for daemon in NameNode DataNode SecondaryNameNode
do
  jps | awk '{print $2}' | grep -qx "$daemon"
done
echo "HDFS daemons: PASS"
hdfs dfsadmin -safemode get | grep -qi "Safe mode is OFF"
echo "HDFS safe mode: OFF"
hdfs dfs -test -d /data/ghsl/curated/cities
echo "Curated city root: PASS"
command -v spark-submit >/dev/null
echo "spark-submit: PASS"
python3 - <<'PY'
import importlib.util
missing=[x for x in ["pyspark","sedona"] if importlib.util.find_spec(x) is None]
if missing: raise RuntimeError("Missing packages: "+repr(missing))
print("PySpark import: PASS")
print("Sedona import: PASS")
PY
echo "HDFS modified: NO"
echo "Spark job launched: NO"
echo "SIX_CITY_MODELING_PREFLIGHT_SUCCESS"'''

r=run_remote(cmd,timeout=300,show_output=True)
print("Return code:",r.returncode)
print("CELL 3: SUCCESS")

In [ ]:
# ============================================================
# CELL 4: DISCOVER EXACT HDFS CITY SLUGS AND ALL 60 PATHS
# ============================================================

cmd=r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
python3 - <<'PY'
import json,re,subprocess
root="/data/ghsl/curated/cities"
epochs=[1975,1980,1985,1990,1995,2000,2005,2010,2015,2020]
r=subprocess.run(["hdfs","dfs","-ls",root],text=True,capture_output=True)
if r.returncode: raise RuntimeError(r.stderr)
slugs=[]
for line in r.stdout.splitlines():
    m=re.search(r"/city=([^/]+)$",line.split()[-1] if line.split() else "")
    if m: slugs.append(m.group(1))
slugs=sorted(set(slugs))
parts=[]
for slug in slugs:
    base=f"{root}/city={slug}/enriched_grid_year"
    if subprocess.run(["hdfs","dfs","-test","-d",base]).returncode: continue
    for epoch in epochs:
        p=f"{base}/epoch={epoch}"
        parts.append({"city_slug":slug,"epoch":epoch,"path":"hdfs://localhost:9000"+p,"exists":subprocess.run(["hdfs","dfs","-test","-d",p]).returncode==0})
print("INVENTORY_JSON_BEGIN")
print(json.dumps({"all_city_slugs":slugs,"partitions":parts},sort_keys=True))
print("INVENTORY_JSON_END")
PY'''

r=run_remote(cmd,timeout=300,show_output=False)
m=re.search(r"INVENTORY_JSON_BEGIN
(.*?)
INVENTORY_JSON_END",r.stdout,re.S)
if not m:
    print(r.stdout)
    if r.stderr: print("REMOTE STDERR:");print(r.stderr)
    raise RuntimeError("Could not parse inventory JSON.")
RAW_HDFS_INVENTORY=json.loads(m.group(1))
DISCOVERED_CITY_SLUGS=RAW_HDFS_INVENTORY["all_city_slugs"]
print("Discovered HDFS city slugs:")
for s in DISCOVERED_CITY_SLUGS: print(" -",s)
print("CELL 4: SUCCESS")

In [ ]:
# ============================================================
# CELL 5: RESOLVE THE SIX TRAINING CITIES AND VERIFY 60 PATHS
# ============================================================

def resolve_slug(name,hints,slugs):
    exact=[s for s in slugs if any(h.lower() in s.lower() for h in hints)]
    if len(exact)!=1:
        raise RuntimeError(f"Could not uniquely resolve {name}. Hints={hints}, matches={exact}, discovered={slugs}")
    return exact[0]

RESOLVED_CITY_SLUGS={name:resolve_slug(name,hints,DISCOVERED_CITY_SLUGS) for name,hints in TRAINING_CITY_HINTS.items()}
if len(set(RESOLVED_CITY_SLUGS.values()))!=6:
    raise RuntimeError("Resolved city slugs are not unique: "+repr(RESOLVED_CITY_SLUGS))

lookup={(x["city_slug"],int(x["epoch"])):x for x in RAW_HDFS_INVENTORY["partitions"]}
SIX_CITY_PARTITIONS=[]
missing=[]
for name,slug in RESOLVED_CITY_SLUGS.items():
    for epoch in EXPECTED_EPOCHS:
        x=lookup.get((slug,epoch))
        if not x or not x["exists"]:
            missing.append((name,slug,epoch))
        else:
            SIX_CITY_PARTITIONS.append({"city_name":name,"city_slug":slug,"epoch":epoch,"path":x["path"]})
if missing: raise RuntimeError("Missing enriched partitions: "+repr(missing))
if len(SIX_CITY_PARTITIONS)!=60: raise RuntimeError(f"Expected 60 partitions, found {len(SIX_CITY_PARTITIONS)}")

for name,slug in RESOLVED_CITY_SLUGS.items():
    n=sum(x["city_slug"]==slug for x in SIX_CITY_PARTITIONS)
    print(f"{name}: {slug}: {n}/10 PASS")
print("All enriched partitions: 60/60 PASS")
print("Jaipur included: NO")
print("HDFS modified: NO")
print("CELL 5: SUCCESS")

In [ ]:
# ============================================================
# CELL 6: ONE-JOB SPARK AUDIT OF ROWS, KEYS AND SCHEMAS
# ============================================================

payload=base64.b64encode(json.dumps(SIX_CITY_PARTITIONS,sort_keys=True).encode()).decode()
cmd=r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
PAYLOAD=/tmp/six_city_inventory_payload.json
SCRIPT=/tmp/six_city_inventory_audit.py
echo "__PAYLOAD__" | base64 -d > "$PAYLOAD"
cat > "$SCRIPT" <<'PY'
from pyspark.sql import SparkSession,functions as F
import json,sys
records=json.load(open(sys.argv[1]))
if len(records)!=60: raise RuntimeError(f"Expected 60 records, found {len(records)}")
spark=SparkSession.builder.appName("GHSL-Six-City-Inventory-Audit").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
results=[];schemas={};total=0
try:
  for item in records:
    df=spark.read.parquet(item["path"])
    if "grid_cell_id" not in df.columns: raise RuntimeError(f'{item["city_slug"]} {item["epoch"]}: grid_cell_id missing')
    rows=df.count(); keys=df.select("grid_cell_id").distinct().count(); nulls=df.filter(F.col("grid_cell_id").isNull()).count()
    if rows!=4900: raise RuntimeError(f'{item["city_slug"]} {item["epoch"]}: rows={rows}, expected=4900')
    if keys!=4900: raise RuntimeError(f'{item["city_slug"]} {item["epoch"]}: unique keys={keys}, expected=4900')
    if nulls: raise RuntimeError(f'{item["city_slug"]} {item["epoch"]}: null keys={nulls}')
    sig=[{"name":f.name,"type":f.dataType.simpleString(),"nullable":f.nullable} for f in df.schema.fields]
    sk=json.dumps(sig,sort_keys=True);schemas.setdefault(sk,[]).append(f'{item["city_slug"]}:{item["epoch"]}')
    results.append({**item,"row_count":rows,"distinct_grid_cell_id":keys,"columns":df.columns,"coverage_columns":sorted(c for c in df.columns if "coverage" in c.lower()),"schema_signature":sig})
    total+=rows
    print(f'PASS: {item["city_slug"]} {item["epoch"]} rows={rows} keys={keys}')
  if total!=294000: raise RuntimeError(f"Expected 294000 total rows, found {total}")
  out={"partition_count":len(results),"total_rows":total,"schema_variant_count":len(schemas),"schema_groups":list(schemas.values()),"partitions":results}
  print("AUDIT_JSON_BEGIN");print(json.dumps(out,sort_keys=True));print("AUDIT_JSON_END")
  print("SIX_CITY_ROW_AND_KEY_AUDIT_SUCCESS")
finally:
  spark.stop()
PY
spark-submit --master local[1] --driver-memory 4g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=4 --conf spark.default.parallelism=2 --conf spark.sql.adaptive.enabled=false --conf spark.driver.maxResultSize=512m "$SCRIPT" "$PAYLOAD"
rm -f "$SCRIPT" "$PAYLOAD"
echo "HDFS modified: NO"
echo "SIX_CITY_REMOTE_AUDIT_COMPLETE"'''.replace("__PAYLOAD__",payload)

r=run_remote(cmd,timeout=3600,show_output=False)
m=re.search(r"AUDIT_JSON_BEGIN
(.*?)
AUDIT_JSON_END",r.stdout,re.S)
if not m:
    print(r.stdout[-12000:])
    if r.stderr: print("REMOTE STDERR:");print(r.stderr[-4000:])
    raise RuntimeError("Could not parse Spark audit JSON.")
SIX_CITY_AUDIT=json.loads(m.group(1))
print("Partitions audited:",SIX_CITY_AUDIT["partition_count"])
print("Total rows:",SIX_CITY_AUDIT["total_rows"])
print("Schema variants:",SIX_CITY_AUDIT["schema_variant_count"])
print("Duplicate grid keys: 0")
print("Null grid keys: 0")
print("HDFS modified: NO")
print("CELL 6: SUCCESS")

In [ ]:
# ============================================================
# CELL 7: SCHEMA/COVERAGE REPORT AND LOCAL INVENTORY FREEZE
# ============================================================

city_columns={};city_coverage={}
for x in SIX_CITY_AUDIT["partitions"]:
    city_columns.setdefault(x["city_slug"],set()).update(x["columns"])
    city_coverage.setdefault(x["city_slug"],set()).update(x["coverage_columns"])
common=sorted(set.intersection(*city_columns.values()))
all_columns=sorted(set.union(*city_columns.values()))

print("Schema variants:",SIX_CITY_AUDIT["schema_variant_count"])
print("Common columns across all six cities:",len(common))
print("Coverage columns by city:")
for slug in sorted(city_coverage): print(" -",slug,sorted(city_coverage[slug]))

report={
  "training_cities":RESOLVED_CITY_SLUGS,
  "held_out_city":"Jaipur",
  "expected_epochs":EXPECTED_EPOCHS,
  "expected_rows_per_partition":4900,
  "expected_partition_count":60,
  "expected_total_rows":294000,
  "audit":SIX_CITY_AUDIT,
  "schema_compatibility":{
      "all_columns":all_columns,
      "common_columns":common,
      "columns_by_city":{k:sorted(v) for k,v in city_columns.items()},
      "coverage_columns_by_city":{k:sorted(v) for k,v in city_coverage.items()},
      "schema_variant_count":SIX_CITY_AUDIT["schema_variant_count"],
  },
  "created_at_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
  "hdfs_modified":False,
}
path=LOCAL_ARTIFACT_ROOT/"six_city_inventory.json"
path.write_text(json.dumps(report,indent=2,sort_keys=True)+"
",encoding="utf-8")
print("Inventory report:",path)
print("Cities verified: 6/6")
print("Partitions verified: 60/60")
print("Rows verified: 294000")
print("Jaipur included in training: NO")
print("HDFS modified: NO")
print("DAY_1_SIX_CITY_INVENTORY_SUCCESS")
print("CELL 7: SUCCESS")

## Stop gate

After Cell 7, stop and share the complete output of Cells 6 and 7. The next cells will be built from the observed schema report, not assumptions. They will harmonize coverage fields, perform a dry-run six-city union, validate 294,000 rows and zero duplicate keys, and only then write the versioned longitudinal dataset to a new HDFS analytics path.